# 第10回　因果推論入門：RCTと観察研究
## ―― 「効いた」のか、「もともとそういう人が選んだ」だけなのか

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

ここから因果の話に入る。統計学Ⅰで「相関 ≠ 因果」を学んだが、Ⅱでは **どうすれば因果が言えるのか** に踏み込む。鍵となるのは、またしても **独立** ―― ランダム化が「処置」を「交絡」から切り離す。▶ を上から押そう。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
print("準備OK。次のセルへ。")

---
## 1. 直感クイズ ―― サプリは効いたのか？

あるサプリメントの利用者を調べたら、**飲んでいる人のほうが健康だった**。

**問い：これは「サプリが健康にする」という因果の証拠になるか？**

なりそうに見える。だが ―― サプリを自分から飲む人は、そもそも **健康志向が高い**（運動する・食事に気をつける）人が多いのではないか？　だとすれば、健康なのはサプリのおかげではなく、**もともとそういう人が飲んでいるだけ** かもしれない。

この「健康志向」のように、**原因の候補と結果の両方に影響する隠れた要因**を **交絡（confounder）** と呼ぶ。

---
## 2. 因果とは「反事実」の比較

「サプリが効いた」とは本当はこういう意味だ：

> **同じ人が**、サプリを飲んだ世界と、飲まなかった世界を比べて、健康に差がある。

だが現実には、1人の人について「飲んだ場合」と「飲まなかった場合」の両方は観測できない（これを **反事実** という）。だから私たちは「飲んだ人の集団」と「飲まなかった人の集団」を比べる。

このとき2つの集団が **健康志向などの点でそろっていない** と、比較は因果ではなく交絡を映してしまう。

---
## 3. シミュレーション：交絡が「効果」を捏造する

架空の世界を作る。設定はこうだ ―― **サプリの本当の効果は ゼロ**。健康を決めているのは「健康志向」だけ。そして健康志向が高い人ほど、自分からサプリを飲む。

この世界で「飲んだ人 vs 飲まない人」の健康を比べると、何が見えるか？

In [ ]:
rng = np.random.default_rng(10)
N = 5000
健康志向 = rng.normal(0, 1, N)                       # 隠れた交絡因子

# 健康志向が高い人ほど、自分からサプリを飲む（観察研究の世界）
飲む確率 = 1 / (1 + np.exp(-2 * 健康志向))
飲んだ_観察 = rng.random(N) < 飲む確率

真のサプリ効果 = 0.0                                  # ★本当は効果ゼロ★
健康 = 50 + 真のサプリ効果 * 飲んだ_観察 + 8 * 健康志向 + rng.normal(0, 3, N)

観察の差 = 健康[飲んだ_観察].mean() - 健康[~飲んだ_観察].mean()
print(f"サプリの本当の効果： {真のサプリ効果}（ゼロ）")
print(f"観察データでの『飲んだ人 − 飲まない人』の健康差： {観察の差:+.2f}")
print("→ 効果ゼロのはずなのに、大きな差が見える。これは交絡（健康志向）のしわざ。")

**本当は効果ゼロなのに、観察データでは飲んだ人のほうがずっと健康に見える。** これを「サプリが効く」と結論したら、完全な誤りだ。差の正体は、飲む人と飲まない人で **健康志向が偏っている** ことにある。

なぜ偏るのか。観察研究では「飲むかどうか」を本人が決めるので、**飲む・飲まないと健康志向が連動（相関）している**。第6回からの言葉で言えば、**処置（飲む）と交絡（健康志向）が独立でない**のだ。

---
## 4. RCT ―― ランダム化が交絡を断ち切る

では、どうすれば本当の効果（ゼロ）が分かるか。

**くじ引きで、誰がサプリを飲むかを決める**。これが **RCT（ランダム化比較試験）** だ。本人の健康志向と無関係に割り当てるので、サプリを飲むグループと飲まないグループの健康志向が **平均的にそろう**。

同じ世界で、今度はランダムに割り当ててみよう。

In [ ]:
# RCT：健康志向と無関係に、コイン投げでサプリを割り当てる
飲んだ_RCT = rng.random(N) < 0.5                       # 完全にランダム（健康志向と独立）
健康_RCT = 50 + 真のサプリ効果 * 飲んだ_RCT + 8 * 健康志向 + rng.normal(0, 3, N)

RCTの差 = 健康_RCT[飲んだ_RCT].mean() - 健康_RCT[~飲んだ_RCT].mean()

# 2群の健康志向が、観察とRCTでどれだけそろっているか
print("【2群の健康志向の平均】")
print(f"  観察： 飲んだ群 {健康志向[飲んだ_観察].mean():+.2f} / 飲まない群 {健康志向[~飲んだ_観察].mean():+.2f}  ← 大きく偏る")
print(f"  RCT ： 飲んだ群 {健康志向[飲んだ_RCT].mean():+.2f} / 飲まない群 {健康志向[~飲んだ_RCT].mean():+.2f}  ← ほぼ同じ")
print()
print(f"推定されたサプリ効果： 観察 {観察の差:+.2f} ／ RCT {RCTの差:+.2f}（真の効果 0 に近い）")

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(["観察研究\n(本人が選ぶ)", "RCT\n(くじで割当)"], [観察の差, RCTの差],
        color=["#e8503a", "#3949ab"])
plt.axhline(真のサプリ効果, ls="--", color="gray", label="本当の効果 = 0")
plt.ylabel("推定されたサプリの効果")
plt.title("観察研究は交絡で効果を捏造。RCTは真の効果(0)を当てる")
for i, v in enumerate([観察の差, RCTの差]):
    plt.text(i, v + 0.1, f"{v:+.2f}", ha="center")
plt.legend(); plt.show()

**RCTでは、2群の健康志向がそろい、推定された効果はほぼゼロ＝真の効果に一致した。**

ランダム化の威力は、まさに **「処置」を「交絡」から独立にする**ことにある。くじ引きは健康志向を見ないので、飲む群・飲まない群の健康志向は平均的に等しくなる。だから群間の差は、純粋にサプリの効果だけを映す。これがRCTが「因果推論の金字塔」と呼ばれる理由だ。

> 💡 **今期の糸がここでも**
> 
> 第6〜7回では「独立が崩れると推測が壊れる」だった。因果推論はその逆を積極的に使う ―― **ランダム化で“処置と交絡の独立”を人工的に作り出す**ことで、因果を取り出す。独立は、壊れると困るだけでなく、self で作り出せれば強力な武器になる。

---
## 5. 観察研究はダメなのか ―― いや、注意して使う

とはいえ、現実にはRCTができない場面も多い（「20年間タバコを吸う/吸わない」をくじで割り当てるのは不可能・非倫理的）。その場合は観察研究に頼るしかないが、**交絡を疑い、できる限り調整する**必要がある（→第11回）。

観察研究で気をつけるもう一つの罠が **選択バイアス**：そもそも調べる対象の集め方が偏っていると、何を比べても歪む。RCTで使う **盲検・プラセボ**（偽薬）も、「飲んでいる」という意識自体の影響を消すための工夫だ。

> 💬 **因果を見抜く問い**
> 
> 「Xをした人はYだった」と聞いたら問う ―― **「Xをした人としなかった人は、他の点でそろっている？」「くじで決めた？ それとも本人が選んだ？」**。本人が選んだなら、その差は交絡かもしれない。

---
## 今日のまとめ

| ポイント | 中身 |
|---|---|
| 因果 | 反事実（同じ人の“した世界 vs しない世界”）の比較。本来は観測できない |
| 交絡 | 原因候補と結果の両方に影響する隠れ要因。観察研究で効果を捏造する |
| RCT | くじ引きで処置を割当→処置と交絡が**独立**に→真の効果が出る |
| 観察研究 | RCTが無理なときに使うが、交絡・選択バイアスを疑い調整が必要 |

- 観察データの「効いたように見える差」は、交絡の産物かもしれない。
- ランダム化は「処置を交絡から独立にする」操作。だから因果が言える。

> **課題（Moodle）**：シナリオが因果と言えるかの判断（自動採点）＋「RCTか観察研究か／なぜそう言えるか」の記述。詳しくはMoodleの第10回課題を見ること。

> **次回予告**：第11回「交絡と疑似相関」。今日の交絡をさらに深掘り。「アイスと水難事故」「チョコとノーベル賞」――統制（調整）で見かけの相関を見破る。そして全体と層別で結論が逆転する **シンプソンのパラドックス**。